In [3]:
import pandas as pd
import numpy as np

# Loading weather data
df = pd.read_parquet("data/raw/aws_clean_baseline.parquet")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (263160, 9)


,timestamp,station_id,name,lat,lon,elevation_m,temp_c,humidity_pct,pressure_hpa
0,2023-01-01 00:00:00,DEL001,New Delhi (Safdarjung),28.584,77.206,216.0,9.2,90,994.1
1,2023-01-01 01:00:00,DEL001,New Delhi (Safdarjung),28.584,77.206,216.0,8.8,91,994.1
2,2023-01-01 02:00:00,DEL001,New Delhi (Safdarjung),28.584,77.206,216.0,8.2,92,993.7
3,2023-01-01 03:00:00,DEL001,New Delhi (Safdarjung),28.584,77.206,216.0,7.9,93,993.5
4,2023-01-01 04:00:00,DEL001,New Delhi (Safdarjung),28.584,77.206,216.0,7.9,93,993.5


In [4]:
df_eval=df.copy()
df_eval[["temp_c", "pressure_hpa", "humidity_pct"]] = df_eval[["temp_c", "pressure_hpa", "humidity_pct"]].astype(float)#added because all values should be considered float, else vha floaat value dalne me dikkat aegi like in dampening case, hme float values dalni hai in the selected indexes so dtype should be float
df_eval["is_anomaly"]=0
df_eval["anomaly_type"] = "normal"
df_eval["affected_sensor"] = "none"
df_eval.head()

,timestamp,station_id,name,lat,lon,elevation_m,temp_c,humidity_pct,pressure_hpa,is_anomaly,anomaly_type,affected_sensor
0,2023-01-01 00:00:00,DEL001,New Delhi (Safdarjung),28.584,77.206,216.0,9.2,90.0,994.1,0,normal,none
1,2023-01-01 01:00:00,DEL001,New Delhi (Safdarjung),28.584,77.206,216.0,8.8,91.0,994.1,0,normal,none
2,2023-01-01 02:00:00,DEL001,New Delhi (Safdarjung),28.584,77.206,216.0,8.2,92.0,993.7,0,normal,none
3,2023-01-01 03:00:00,DEL001,New Delhi (Safdarjung),28.584,77.206,216.0,7.9,93.0,993.5,0,normal,none
4,2023-01-01 04:00:00,DEL001,New Delhi (Safdarjung),28.584,77.206,216.0,7.9,93.0,993.5,0,normal,none


In [5]:
#adding spikes 
np.random.seed(42)
n_spikes=int(len(df_eval)*0.004)#so adding 0.4 % spikes(roughly) right now
spike_indices = np.random.choice(df_eval.index, size=n_spikes, replace=False)#selecting which indexes to add spikes at
for index in spike_indices:
    sensor=np.random.choice(["temp_c", "pressure_hpa", "humidity_pct"])
    if sensor=="temp_c":
        spiked_difference = np.random.choice([+30.0, +40.0, -25.0])
        df_eval.loc[index, "temp_c"] += spiked_difference
    elif sensor=="humidity_pct":
        df_eval.loc[index,"humidity_pct"]=np.random.choice([130.0,145.0,-10.0])#setting humidity to an impossible value 
    elif sensor =="pressure_hpa":
        spiked_difference=np.random.choice([+80.0, -80.0])
        df_eval.loc[index,"pressure_hpa"]+=spiked_difference
    df_eval.loc[index,"is_anomaly"]=1
    df_eval.loc[index,"anomaly_type"]="spike"
    df_eval.loc[index, "affected_sensor"] = sensor

df_eval["is_anomaly"].value_counts()#is giving 1052 rows which is roghly 0.4% so evrythings fine
df_eval[df_eval["anomaly_type"]=="spike"][["temp_c","pressure_hpa","humidity_pct","affected_sensor"]]



,temp_c,pressure_hpa,humidity_pct,affected_sensor
106,39.1,1000.6,95.0,temp_c
256,58.9,987.4,64.0,temp_c
1027,16.4,913.2,30.0,pressure_hpa
1054,15.5,991.3,-10.0,humidity_pct
1151,46.2,990.6,90.0,temp_c
...,...,...,...,...
262472,20.1,929.8,76.0,pressure_hpa
262711,16.1,930.3,85.0,pressure_hpa
262756,13.9,1009.9,130.0,humidity_pct
262992,15.8,1010.2,-10.0,humidity_pct


In [6]:
# Injecting Frozen (Stuck) Sensors
np.random.seed(42) 
n_frozen_events = 20 
duration_hours = 12 #frozen for how many hours

stations = df_eval["station_id"].unique()

for i in range(n_frozen_events):
    
    station = np.random.choice(stations)#station chunega
    station_indices = df_eval[df_eval["station_id"] == station].index.to_numpy()#getting the station data, index here is basically hoir by hour data

    if len(station_indices) > duration_hours:
        
        start_pos = np.random.randint(0, len(station_indices) - duration_hours)#so that actual me duration hours tak ka data available ho
        event_idx = station_indices[start_pos : start_pos + duration_hours]
        
        
        sensor = np.random.choice(["temp_c", "pressure_hpa", "humidity_pct"])#selecting the sensor to freeze
     #currently assuming ki value remains feezed exactly , that is no single variation throighout the timeline but small variations ke liye well add secodn derivative later on
        frozen_val = df_eval.loc[event_idx[0], sensor]#overriting the first hour value at every next hour in diuration
        df_eval.loc[event_idx, sensor] = frozen_val
    
        df_eval.loc[event_idx, "is_anomaly"] = 1
        df_eval.loc[event_idx, "anomaly_type"] = "frozen_sensor"
        df_eval.loc[event_idx, "affected_sensor"] = sensor
df_eval["anomaly_type"].value_counts()

anomaly_type
normal           261868
spike              1052
frozen_sensor       240
Name: count, dtype: int64

In [7]:
#now adding the second derivative based anomolies
#basically, assume we have an insulation issue, that is there is something which is blocking sensor's contact wiht the external environment, be it a random object stuck on it like a clot, or maybe shadow due to a huge object kept in front of it ir maybe dust/uce accumulation
# in all such cases, the effect of external factors on changing the poarameters will reduce drastically, so say temp was to be increased due to sun but cloth got stuck, it wiull still increase but as cloth will boco most of rays, the incerase will be really small, so the graph will become largely linerly increasing (because the factor that was causing the change got supressed so the changes are negligible ) and hence the second derivative will be very clkose to zero
np.random.seed(42)
n_linear_events = 20
linear_duration_hours = 3#3 ghante ke liye insulation fault rha(this can be changed)
stations = df_eval["station_id"].unique()
for i in range(n_linear_events):
    station = np.random.choice(stations)
    station_indices = df_eval[df_eval["station_id"] == station].index.to_numpy()
    
    if len(station_indices) > linear_duration_hours:
        start_pos = np.random.randint(0, len(station_indices) - linear_duration_hours)
        event_idx = station_indices[start_pos : start_pos + linear_duration_hours]
        sensor = np.random.choice(["temp_c", "pressure_hpa", "humidity_pct"])
        start_val = df_eval.loc[event_idx[0], sensor]
        end_val = start_val + np.random.choice([1.0, -1.0, 0.5, -0.5])#for now i am setting te final value inc?dec by 1 or half unit but we can imporve this further by making iut sensor specific
        perfect_straight_line = np.linspace(start_val, end_val, num=linear_duration_hours)#presently, keeping it purely linear but we can set thresholds for second derivative later on in case close to linear ho
        df_eval.loc[event_idx, sensor] = (perfect_straight_line)
        df_eval.loc[event_idx, "is_anomaly"] = 1
        df_eval.loc[event_idx, "anomaly_type"] = "linear_dampened"
        df_eval.loc[event_idx, "affected_sensor"] = sensor
df_eval["anomaly_type"].value_counts()

anomaly_type
normal             261814
spike                1052
frozen_sensor         234
linear_dampened        60
Name: count, dtype: int64

now adding the gradual drifts(aging sensor data)
now the aging sensor reading is basically superpostion of a linear graph(due to aging)+ the actual noise graph(the actual reading graph)

In [8]:
np.random.seed(42)
n_drift_events = 15
#lets assume each drift continues for later 10 days so a total of 240 hours, obviously we should technically increase this as aging will continue but for now keeping the duration of 10 days
drift_duration_hours = 240
stations = df_eval["station_id"].unique()
for i in range(n_drift_events):
    station = np.random.choice(stations)
    station_indices = df_eval[df_eval["station_id"] == station].index.to_numpy()
    
    if len(station_indices) > drift_duration_hours:
        start_pos = np.random.randint(0, len(station_indices) - drift_duration_hours)
        event_idx = station_indices[start_pos : start_pos + drift_duration_hours]
        sensor = np.random.choice(["temp_c", "pressure_hpa","humidity_pct"])
        #presently assuming the final error casued due to aging is one of the below values(again ve should chnage it alter on by actual data on how fast each sensor degrades)
        total_error = np.random.choice([2.5, -2.5, 3.0, -3.0])
        drift_array = np.linspace(0, total_error, num=drift_duration_hours)#the aging line
        df_eval.loc[event_idx, sensor] += drift_array#superposition of normal reading graph and aging line 

        df_eval.loc[event_idx, "is_anomaly"] = 1
        df_eval.loc[event_idx, "anomaly_type"] = "gradual_drift"
        df_eval.loc[event_idx, "affected_sensor"] = sensor
df_eval[df_eval["anomaly_type"]=="gradual_drift"]["affected_sensor"].value_counts()





affected_sensor
temp_c          1440
humidity_pct    1200
pressure_hpa     960
Name: count, dtype: int64

now adding the missing values
the thing is that when there is a power cut, or maybe a wire is cut/removed due to any reason or dust gets accumulated over the solar panel or the battery gets drained mid night and no way to power up the sensor, in all such cases there will be a flow of missing values as sensor wont report any reading


In [9]:
np.random.seed(42)
n_dropout_events = 25 
dropout_duration_hours = 6

stations = df_eval["station_id"].unique()
for i in range(n_dropout_events):
    station = np.random.choice(stations)
    station_indices = df_eval[df_eval["station_id"] == station].index.to_numpy()
    if len(station_indices) > dropout_duration_hours:
        start_pos = np.random.randint(0, len(station_indices) - dropout_duration_hours)
        event_idx = station_indices[start_pos : start_pos + dropout_duration_hours]
        #now in case of power cut, the whole station will die but in case of a wire cut/supply fault to a specific sensor, only that sensor will die
        #for now, lets assume both events as equally probable(50%-50%)
        outage_type = np.random.choice(["whole_station", "single_sensor"])#random so almost close to 50%
        if outage_type == "whole_station":
            df_eval.loc[event_idx, ["temp_c", "pressure_hpa", "humidity_pct"]] = np.nan
            affected = "all_sensors"
        else:
            sensor = np.random.choice(["temp_c", "pressure_hpa", "humidity_pct"])
            df_eval.loc[event_idx, sensor] = np.nan
            affected = sensor
        df_eval.loc[event_idx, "is_anomaly"] = 1
        df_eval.loc[event_idx, "anomaly_type"] = "dropout_power_cut"
        df_eval.loc[event_idx, "affected_sensor"] = affected
df_eval[["anomaly_type","affected_sensor"]].value_counts()

anomaly_type       affected_sensor
normal             none               258252
gradual_drift      temp_c               1434
                   humidity_pct         1188
                   pressure_hpa          960
spike              humidity_pct          350
                   pressure_hpa          347
                   temp_c                338
dropout_power_cut  all_sensors            66
frozen_sensor      temp_c                 48
                   humidity_pct           42
                   pressure_hpa           36
dropout_power_cut  temp_c                 30
                   pressure_hpa           30
                   humidity_pct           24
linear_dampened    pressure_hpa            9
                   temp_c                  3
                   humidity_pct            3
Name: count, dtype: int64

now adding cross sensor decoupling
really important so understand pls:
in india, temp of 50 degree celsius is possible in etreme summer and humidity of 90% is also possible in monsoon 
but the combination of 90% humidity at 50 degree celcius is practically not possible as hotter air has more capacoty to hold moisture so relaive humidity becomes less
in such cases, no individual reading is impossible but the simultaneous combo of these reading confirms the anomaly 
this will be solved by the physics layer

In [10]:
np.random.seed(42)
n_decoupling_events = 15
decoupling_duration_hours = 8
stations = df_eval["station_id"].unique()
for i in range(n_decoupling_events):
    station = np.random.choice(stations)
    station_indices = df_eval[df_eval["station_id"] == station].index.to_numpy()
    
    if len(station_indices) > decoupling_duration_hours:
        start_pos = np.random.randint(0, len(station_indices) - decoupling_duration_hours)
        event_idx = station_indices[start_pos : start_pos + decoupling_duration_hours]
        df_eval.loc[event_idx, "temp_c"] = np.random.choice([48.0, 50.0, 52.0])
        df_eval.loc[event_idx, "humidity_pct"] = np.random.choice([95.0, 98.0, 99.0])


        df_eval.loc[event_idx, "is_anomaly"] = 1
        df_eval.loc[event_idx, "anomaly_type"] = "cross_sensor_decoupling"
        df_eval.loc[event_idx, "affected_sensor"] = "temp_humidity"#as one of these will be affected, now which one exactly can be found via further layers like chronos 2
df_eval["anomaly_type"].value_counts()


anomaly_type
normal                     258231
gradual_drift                3530
spike                        1034
frozen_sensor                 122
cross_sensor_decoupling       120
dropout_power_cut             108
linear_dampened                15
Name: count, dtype: int64

In [11]:
#the decoupling btw temp and pressure will be included later below as it would need altitude level also

Now the high frequency noise burst
this happens when a wire has a loose connection, the sensor isn't properly shielded from electromagnetic interference (EMI) from a nearby radio tower, or strong wind physically vibrates the sensor housing.

Instead of a smooth natural curve, the data suddenly gets extremely "jittery" for a few hours. The values wildly jump up and down around the true value.

To add such anomolies, ill be selecting some random numbers centred at 0 with high standard deviation to assure high random deviation from centre 0

In [12]:
np.random.seed(42)
n_noise_events = 20
noise_duration_hours = 10#assuming jittering lasts 10 hours

stations = df_eval["station_id"].unique()

for i in range(n_noise_events):
    station = np.random.choice(stations)
    station_indices = df_eval[df_eval["station_id"] == station].index.to_numpy()
    
    if len(station_indices) > noise_duration_hours:
        start_pos = np.random.randint(0, len(station_indices) - noise_duration_hours)
        event_idx = station_indices[start_pos : start_pos + noise_duration_hours]
        
        sensor = np.random.choice(["temp_c", "pressure_hpa", "humidity_pct"])
        #lets keep standard deviation of 5, so itll give random jumps about 0 like +4.2,-6,+1.5etc(basically averrage distance from 0 will be 5 units)
        noise = np.random.normal(loc=0.0, scale=5.0, size=noise_duration_hours)
        #scale is just the standard deviation(conventional name) and loc is the centre/mean 
        #normal here refers to gaussian distribution, which means that the noise is centred at the origin and follows a bell-shaped curve about it
        #is this bell model, the distribution is 68.2%,95.4% and 99.7% for deviation of std, 2std and 3std from centre
        #this means that since our std=5,68.2%values will fall in displacement of 0(centre)+-5(std), and similarly +-10 and +-15 for 95.4 and 99.7% data
        
        df_eval.loc[event_idx, sensor] += noise#adding the noise over usual data
        df_eval.loc[event_idx, "is_anomaly"] = 1
        df_eval.loc[event_idx, "anomaly_type"] = "noise_burst"
        df_eval.loc[event_idx, "affected_sensor"] = sensor
df_eval["anomaly_type"].value_counts()

anomaly_type
normal                     258069
gradual_drift                3508
spike                        1032
noise_burst                   200
frozen_sensor                 122
cross_sensor_decoupling       112
dropout_power_cut             102
linear_dampened                15
Name: count, dtype: int64

Now the last anomaly-- Impractical Pressure :
There is a norm in weather calculations, the readings used for Weather maps and related stuff is Mean Sea Level Pressure at a location, this is because a low pressure at high altitude centre , say at shimla, isnt an alert so there is no need to mark it red area in map, thats why sensors are calliberated using a data loger which convertts the raw reading at that altitude to that at mean sea level using a formula(hypsometric eqn) and sends that for weather alert maps/systems

Now there is a possibility that the technichian might by mistake make improper connection while maintenance due to which Pmsl gets sent to the Pressure_hpa column(which is raw pressure and not adjusted) or maybe someone resets the datalogger due to which it starts sending raw pressure as Pmsl without the copnversion as by default the altitude level set in datalogggers in 0 so we need to change that else math wont convert it into actual Pmsl

In all such cases, the Raw Pressure that we need is given the value fo mean sea level at that altitude but weell use this to calculate the mean sea leevl pressure, so using a mean sea level pressure to find a mean sea level pressure by math will give a further higher pressure which will be suspiciously bigger than the usual mean sea level pressur(1013.25hpa)
btw hpa is hectopascal and 1013.25hpa=1atm 


In [13]:
np.random.seed(42)
n_barometric_events = 15
barometric_duration_hours = 12
# Targetting stations with an elevation higher than 500 meters(and hence low pressure)
high_stations = df_eval[df_eval["elevation_m"] > 500]["station_id"].unique()
#in case no high stations exist, still adding the anomolies for model(though of not practical use but for demonstation purpose), though we should change this later and simply select the stations with highest altitudes available in such cases or maybe lower the threshold altitude
if len(high_stations) == 0:
    high_stations = df_eval["station_id"].unique()
for i in range(n_barometric_events):
    station = np.random.choice(high_stations)
    station_indices = df_eval[df_eval["station_id"] == station].index.to_numpy()
    
    if len(station_indices) > barometric_duration_hours:
        start_pos = np.random.randint(0, len(station_indices) - barometric_duration_hours)
        event_idx = station_indices[start_pos : start_pos + barometric_duration_hours]
        #fault-1, a high altitude sensor reports usual sea level pressure(about 1014hpa)
        #due to this, when well calculate Pmsl, its basically calutaing a Pmsl using a Pmsl instead of P_actual_at_altitude actual giving a further high Pmsl
        df_eval.loc[event_idx, "pressure_hpa"] = np.random.choice([1013.25, 1015.0, 1010.5])


        df_eval.loc[event_idx, "is_anomaly"] = 1
        df_eval.loc[event_idx, "anomaly_type"] = "barometric_altitude_inconsistency"
        df_eval.loc[event_idx, "affected_sensor"] = "pressure_hpa"
df_eval["anomaly_type"].value_counts()
        

anomaly_type
normal                               257889
gradual_drift                          3508
spike                                  1032
noise_burst                             200
barometric_altitude_inconsistency       180
frozen_sensor                           122
cross_sensor_decoupling                 112
dropout_power_cut                       102
linear_dampened                          15
Name: count, dtype: int64

In [ ]:
#saving the anomaly data into seperate parquet so that we can pivot to faster methods instead of pandas
df_eval.to_parquet("aws_evaluation_dataset.parquet", index=False)


,timestamp,station_id,name,lat,lon,elevation_m,temp_c,humidity_pct,pressure_hpa,is_anomaly,anomaly_type,affected_sensor
